<a href="https://colab.research.google.com/github/Kayla-afk/Tugas-Kuliah-D4-Sains-Data-Terapan/blob/main/Tugas_1_Sistem_Rekomendasi_Content_Based_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import Library & Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# Dataset
data = {
    "Product_ID": ["P001","P002","P003","P004","P005","P006","P007","P008","P009","P010"],
    "Nama": ["Laptop A","Smartphone B","Headphone C","Meja Kayu","Kursi Kantor",
             "Blender D","Kipas Angin E","Jam Tangan F","Sepatu Sport G","Kompor Gas H"],
    "Kategori": ["Elektronik","Elektronik","Aksesoris","Furniture","Furniture",
                 "Dapur","Elektronik","Fashion","Fashion","Dapur"],
    "Harga": [7500000,3200000,850000,1200000,1500000,700000,500000,450000,650000,1000000],
    "Stok": [10,25,15,8,12,20,30,18,22,10],
    "Rating": [4.5,4.7,4.3,4.6,4.4,4.2,4.0,4.1,4.5,4.3]
}

df = pd.DataFrame(data)
df

,Product_ID,Nama,Kategori,Harga,Stok,Rating
0,P001,Laptop A,Elektronik,7500000,10,4.5
1,P002,Smartphone B,Elektronik,3200000,25,4.7
2,P003,Headphone C,Aksesoris,850000,15,4.3
3,P004,Meja Kayu,Furniture,1200000,8,4.6
4,P005,Kursi Kantor,Furniture,1500000,12,4.4
5,P006,Blender D,Dapur,700000,20,4.2
6,P007,Kipas Angin E,Elektronik,500000,30,4.0
7,P008,Jam Tangan F,Fashion,450000,18,4.1
8,P009,Sepatu Sport G,Fashion,650000,22,4.5
9,P010,Kompor Gas H,Dapur,1000000,10,4.3


### Step 1: Preprocessing

In [2]:
#1. Normalisasi (Min-Max Scaling)
scaler = MinMaxScaler()

df[['Harga_norm','Stok_norm','Rating_norm']] = scaler.fit_transform(
    df[['Harga','Stok','Rating']]
)

df

,Product_ID,Nama,Kategori,Harga,Stok,Rating,Harga_norm,Stok_norm,Rating_norm
0,P001,Laptop A,Elektronik,7500000,10,4.5,1.000000,0.090909,0.714286
1,P002,Smartphone B,Elektronik,3200000,25,4.7,0.390071,0.772727,1.000000
2,P003,Headphone C,Aksesoris,850000,15,4.3,0.056738,0.318182,0.428571
3,P004,Meja Kayu,Furniture,1200000,8,4.6,0.106383,0.000000,0.857143
4,P005,Kursi Kantor,Furniture,1500000,12,4.4,0.148936,0.181818,0.571429
5,P006,Blender D,Dapur,700000,20,4.2,0.035461,0.545455,0.285714
6,P007,Kipas Angin E,Elektronik,500000,30,4.0,0.007092,1.000000,0.000000
7,P008,Jam Tangan F,Fashion,450000,18,4.1,0.000000,0.454545,0.142857
8,P009,Sepatu Sport G,Fashion,650000,22,4.5,0.028369,0.636364,0.714286
9,P010,Kompor Gas H,Dapur,1000000,10,4.3,0.078014,0.090909,0.428571


In [3]:
#2. One Hot Encoding (Kategori)
df_ohe = pd.get_dummies(df['Kategori'], prefix='Kategori')

df = pd.concat([df, df_ohe], axis=1)
df

,Product_ID,Nama,Kategori,Harga,Stok,Rating,Harga_norm,Stok_norm,Rating_norm,Kategori_Aksesoris,Kategori_Dapur,Kategori_Elektronik,Kategori_Fashion,Kategori_Furniture
0,P001,Laptop A,Elektronik,7500000,10,4.5,1.000000,0.090909,0.714286,False,False,True,False,False
1,P002,Smartphone B,Elektronik,3200000,25,4.7,0.390071,0.772727,1.000000,False,False,True,False,False
2,P003,Headphone C,Aksesoris,850000,15,4.3,0.056738,0.318182,0.428571,True,False,False,False,False
3,P004,Meja Kayu,Furniture,1200000,8,4.6,0.106383,0.000000,0.857143,False,False,False,False,True
4,P005,Kursi Kantor,Furniture,1500000,12,4.4,0.148936,0.181818,0.571429,False,False,False,False,True
5,P006,Blender D,Dapur,700000,20,4.2,0.035461,0.545455,0.285714,False,True,False,False,False
6,P007,Kipas Angin E,Elektronik,500000,30,4.0,0.007092,1.000000,0.000000,False,False,True,False,False
7,P008,Jam Tangan F,Fashion,450000,18,4.1,0.000000,0.454545,0.142857,False,False,False,True,False
8,P009,Sepatu Sport G,Fashion,650000,22,4.5,0.028369,0.636364,0.714286,False,False,False,True,False
9,P010,Kompor Gas H,Dapur,1000000,10,4.3,0.078014,0.090909,0.428571,False,True,False,False,False


### Step 2: Feature Extraction


In [4]:
fitur = df[['Harga_norm','Stok_norm','Rating_norm'] + list(df_ohe.columns)]
fitur

,Harga_norm,Stok_norm,Rating_norm,Kategori_Aksesoris,Kategori_Dapur,Kategori_Elektronik,Kategori_Fashion,Kategori_Furniture
0,1.000000,0.090909,0.714286,False,False,True,False,False
1,0.390071,0.772727,1.000000,False,False,True,False,False
2,0.056738,0.318182,0.428571,True,False,False,False,False
3,0.106383,0.000000,0.857143,False,False,False,False,True
4,0.148936,0.181818,0.571429,False,False,False,False,True
5,0.035461,0.545455,0.285714,False,True,False,False,False
6,0.007092,1.000000,0.000000,False,False,True,False,False
7,0.000000,0.454545,0.142857,False,False,False,True,False
8,0.028369,0.636364,0.714286,False,False,False,True,False
9,0.078014,0.090909,0.428571,False,True,False,False,False


### Step 3: Entropy (Feature Selection)

In [5]:
#Hitung distribusi kategori
kategori_counts = df['Kategori'].value_counts()
total = len(df)

prob = kategori_counts / total
entropy = -np.sum(prob * np.log2(prob))

print("Distribusi:\n", kategori_counts)
print("\nProbabilitas:\n", prob)
print("\nEntropy:", entropy)

Distribusi:
 Kategori
Elektronik    3
Furniture     2
Dapur         2
Fashion       2
Aksesoris     1
Name: count, dtype: int64

Probabilitas:
 Kategori
Elektronik    0.3
Furniture     0.2
Dapur         0.2
Fashion       0.2
Aksesoris     0.1
Name: count, dtype: float64

Entropy: 2.2464393446710154


### Step 4: Feature Weighting


In [7]:
jumlah_kategori = len(kategori_counts)
weight = entropy / np.log2(jumlah_kategori)

print("Weight:", weight)

#Terapkan bobot ke fitur kategori
fitur_weighted = fitur.copy()

for col in df_ohe.columns:
    fitur_weighted[col] = fitur_weighted[col] * weight

fitur_weighted

Weight: 0.9674887648835616


,Harga_norm,Stok_norm,Rating_norm,Kategori_Aksesoris,Kategori_Dapur,Kategori_Elektronik,Kategori_Fashion,Kategori_Furniture
0,1.000000,0.090909,0.714286,0.000000,0.000000,0.967489,0.000000,0.000000
1,0.390071,0.772727,1.000000,0.000000,0.000000,0.967489,0.000000,0.000000
2,0.056738,0.318182,0.428571,0.967489,0.000000,0.000000,0.000000,0.000000
3,0.106383,0.000000,0.857143,0.000000,0.000000,0.000000,0.000000,0.967489
4,0.148936,0.181818,0.571429,0.000000,0.000000,0.000000,0.000000,0.967489
5,0.035461,0.545455,0.285714,0.000000,0.967489,0.000000,0.000000,0.000000
6,0.007092,1.000000,0.000000,0.000000,0.000000,0.967489,0.000000,0.000000
7,0.000000,0.454545,0.142857,0.000000,0.000000,0.000000,0.967489,0.000000
8,0.028369,0.636364,0.714286,0.000000,0.000000,0.000000,0.967489,0.000000
9,0.078014,0.090909,0.428571,0.000000,0.967489,0.000000,0.000000,0.000000


In [ ]:
#Content-Based Filtering (Similarity) menggunakan Cosine Similarity

In [8]:
similarity = cosine_similarity(fitur_weighted)

sim_df = pd.DataFrame(similarity, index=df['Nama'], columns=df['Nama'])
sim_df

Nama,Laptop A,Smartphone B,Headphone C,Meja Kayu,Kursi Kantor,Blender D,Kipas Angin E,Jam Tangan F,Sepatu Sport G,Kompor Gas H
Nama,,,,,,,,,,
Laptop A,1.000000,0.822121,0.226020,0.353674,0.318950,0.160846,0.474342,0.084851,0.279739,0.235196
Smartphone B,0.822121,1.000000,0.384192,0.422835,0.409336,0.383494,0.750630,0.279586,0.545764,0.303282
Headphone C,0.226020,0.384192,1.000000,0.260205,0.245015,0.234756,0.206939,0.172519,0.338851,0.184193
Meja Kayu,0.353674,0.422835,0.260205,1.000000,0.968329,0.167111,0.000418,0.087546,0.348595,0.271983
Kursi Kantor,0.318950,0.409336,0.245015,0.968329,1.000000,0.203261,0.114489,0.132694,0.338036,0.223353
Blender D,0.160846,0.383494,0.234756,0.167111,0.203261,1.000000,0.341818,0.233357,0.353647,0.909141
Kipas Angin E,0.474342,0.750630,0.206939,0.000418,0.114489,0.341818,1.000000,0.302911,0.336171,0.061725
Jam Tangan F,0.084851,0.279586,0.172519,0.087546,0.132694,0.233357,0.302911,1.000000,0.904396,0.089290
Sepatu Sport G,0.279739,0.545764,0.338851,0.348595,0.338036,0.353647,0.336171,0.904396,1.000000,0.252676
